In [13]:
import json
import pandas as pd
import h5py
import numpy as np
import tensorflow as tf
import keras
from keras.layers import Input, Dense, Dropout, LSTM, Flatten, GRU,TimeDistributed, Conv1D, BatchNormalization
from keras.models import Model, Sequential
from keras.optimizers import Adam, SGD
from sklearn.model_selection import train_test_split
import os
import h5py
import matplotlib.pyplot as plt
from keras import regularizers
from tensorflow.keras.regularizers import l1
import ast
from tqdm import tqdm
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import uniform, truncnorm, randint
from sklearn.metrics import make_scorer
from sklearn.metrics import accuracy_score, precision_score, recall_score
from sklearn.model_selection import RepeatedStratifiedKFold
from scipy.stats import loguniform
from pandas import read_csv
# from keras.wrappers.scikit_learn import KerasClassifier
from scikeras.wrappers import KerasClassifier
from sklearn.metrics import roc_auc_score
from qkeras import *
import qkeras
from tensorflow.keras.models import load_model
from qkeras.utils import model_quantize
from qkeras.utils import model_save_quantized_weights

import matplotlib.pyplot as plt

## Load Training and Testing data

In [14]:
def makeRoc(features_val, labels_val, labels, model, outputDir='', outputSuffix=''):
    from sklearn.metrics import roc_curve, auc
    labels_pred = model.predict(features_val)
    df = pd.DataFrame()
    fpr = {}
    tpr = {}
    auc1 = {}
    plt.figure(figsize=(10,8))       
    for i, label in enumerate(labels):

        df[label] = labels_val[:,i]
        df[label + '_pred'] = labels_pred[:,i]
        fpr[label], tpr[label], threshold = roc_curve(df[label],df[label+'_pred'])
        auc1[label] = auc(fpr[label], tpr[label])
        plt.plot(fpr[label],tpr[label],label='%s tagger, AUC = %.1f%%'%(label.replace('j_',''),auc1[label]*100.))
        
    plt.plot([0, 1], [0, 1], lw=1, color='black', linestyle='--')
    #plt.semilogy()
    plt.xlabel("Background Efficiency")
    plt.ylabel("Signal Efficiency")
    plt.xlim([-0.05, 1.05])
    plt.ylim(0.001,1.05)
    plt.grid(True)
    plt.legend(loc='lower right')
    plt.figtext(0.25, 0.90,'LSTM ROC Curve',fontweight='bold', wrap=True, horizontalalignment='right', fontsize=14)
    #plt.figtext(0.35, 0.90,'preliminary', style='italic', wrap=True, horizontalalignment='center', fontsize=14) 
    #plt.savefig('%sROC_%s.pdf'%(outputDir, outputSuffix))
    return labels_pred

In [15]:
def learningCurve(history):
    plt.figure(figsize=(10,8))
    plt.plot(history.history['loss'], linewidth=1)
    plt.plot(history.history['val_loss'], linewidth=1)
    plt.title('Model Loss over Epochs')
    plt.ylabel('Loss')
    plt.xlabel('Epoch')
    plt.legend(['training sample loss','validation sample loss'])
    #plt.savefig('Learning_curve.pdf')
    plt.show()
    plt.close()

In [16]:
# Load and format data
X_train = np.load('X_train.npy', allow_pickle=True)
X_test = np.load('X_test.npy', allow_pickle=True)
X_valid = np.load('X_valid.npy', allow_pickle=True)
y_train = np.load('y_train.npy', allow_pickle=True)
y_test = np.load('y_test.npy', allow_pickle=True)
y_valid = np.load('y_valid.npy', allow_pickle=True)

print(X_train.shape)

X_trainzero = np.zeros((X_train.shape[0],100,3))

for x in tqdm(range(len(X_train))):
#     print(len(X_train[x]))
  for y in range(100):
    for z in range(3):
      if y >= len(X_train[x]):
        break
      else:
        X_trainzero[x][y][z] = X_train[x][y][z]

X_validzero = np.zeros((X_valid.shape[0],100,3))

for x in tqdm(range(len(X_valid))):
#     print(len(X_train[x]))
  for y in range(100):
    for z in range(3):
      if y >= len(X_valid[x]):
        break
      else:
        X_validzero[x][y][z] = X_valid[x][y][z]

X_trainzero = np.concatenate((X_trainzero, X_validzero)) # combine training + validation
        
y_labhot = np.zeros((len(y_train),5))

y_labhot.shape

num = 0
for x in y_train:
  if x == 0:
    y_labhot[num][0] = 1
  elif x == 1: 
    y_labhot[num][1] = 1
  elif x == 2: 
    y_labhot[num][2] = 1
  elif x == 3: 
    y_labhot[num][3] = 1
  elif x == 4: 
    y_labhot[num][4] = 1
  num = num + 1

y_labhot

y_vlabhot = np.zeros((len(y_valid),5))

y_vlabhot.shape

num = 0
for x in y_valid:
  if x == 0:
    y_vlabhot[num][0] = 1
  elif x == 1: 
    y_vlabhot[num][1] = 1
  elif x == 2: 
    y_vlabhot[num][2] = 1
  elif x == 3: 
    y_vlabhot[num][3] = 1
  elif x == 4: 
    y_vlabhot[num][4] = 1
  num = num + 1

y_labhot = np.concatenate((y_labhot, y_vlabhot)) # combine training + validation

X_testzero = np.zeros((len(X_test),100,3))

X_testzero[0][0][0]

for x in tqdm(range(len(X_test))):
  for y in range(100):
    for z in range(3):
      if y >= len(X_test[x]):
        break
      else:
        X_testzero[x][y][z] = X_test[x][y][z]
        
y_tlabhot = np.zeros((len(y_test),5))

num = 0
for x in y_test:
  if x == 0:
    y_tlabhot[num][0] = 1
  elif x == 1: 
    y_tlabhot[num][1] = 1
  elif x == 2: 
    y_tlabhot[num][2] = 1
  elif x == 3: 
    y_tlabhot[num][3] = 1
  elif x == 4: 
    y_tlabhot[num][4] = 1
  num = num + 1

(537009,)


100%|██████████| 12500/12500 [00:02<00:00, 4848.08it/s]


In [17]:
shuffler = np.random.permutation(len(X_trainzero))
array1_shuffled = X_trainzero[shuffler]
array2_shuffled = y_labhot[shuffler]

## Models

## LSTM

In [ ]:
# Inputs = Input(shape = (100,3))
# x = Conv1D(128,(6))(Inputs)
# x = Activation('relu', name = 'relu_0')(x)
# x = Conv1D(64,(3))(x)
# x = Activation('relu', name = 'relu_1')(x)
# x = Conv1D(64, (1))(x)
# x = Activation('relu', name = 'relu_2')(x)
# x = LSTM(128, return_sequences=True, name='lstm_10')(x)
# x = Dropout(rate = 0.5)(x)
# x = LSTM(64, return_sequences=False, name='lstm_11')(x)
# predictions = Dense(5, kernel_initializer='lecun_uniform', name='rnn_densef')(x)
# predictions = Activation('softmax', name = 'output_softmax')(predictions)
# lstm = Model(inputs=Inputs, outputs=predictions)

# lstm.summary()

Model: "model_2"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_3 (InputLayer)        [(None, 100, 3)]          0         
                                                                 
 conv1d_3 (Conv1D)           (None, 95, 128)           2432      
                                                                 
 relu_0 (Activation)         (None, 95, 128)           0         
                                                                 
 conv1d_4 (Conv1D)           (None, 93, 64)            24640     
                                                                 
 relu_1 (Activation)         (None, 93, 64)            0         
                                                                 
 conv1d_5 (Conv1D)           (None, 93, 64)            4160      
                                                                 
 relu_2 (Activation)         (None, 93, 64)            0   

## GRU

In [ ]:
# Inputs = Input(shape = (100,3))

# x = GRU(32, return_sequences=True)(Inputs)
# x = Flatten()(x)
# x = Dense(32)(x)
# x = Activation('relu', name = 'relu_0')(x)
# x = Dropout(rate = 0.5)(x)
# x = Dense(16)(x)
# x = Activation('relu', name = 'relu_1')(x)
# predictions = Dense(5, kernel_initializer='lecun_uniform', name='rnn_densef')(x)
# predictions = Activation('softmax', name = 'softmax')(predictions)
# gru = Model(inputs=Inputs, outputs=predictions)

# gru.summary()

Model: "model_3"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_4 (InputLayer)        [(None, 100, 3)]          0         
                                                                 
 gru_1 (GRU)                 (None, 100, 32)           3552      
                                                                 
 flatten_1 (Flatten)         (None, 3200)              0         
                                                                 
 dense_2 (Dense)             (None, 32)                102432    
                                                                 
 relu_0 (Activation)         (None, 32)                0         
                                                                 
 dropout_3 (Dropout)         (None, 32)                0         
                                                                 
 dense_3 (Dense)             (None, 16)                528 

## LSTM Floating Point Model Training

In [20]:
# adam = Adam(lr = 0.001)
# lstm.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
# history = lstm.fit(array1_shuffled, array2_shuffled, batch_size = 512, epochs = 50, 
#                     validation_split = 0.2, shuffle = True, callbacks = None,
#                     use_multiprocessing=True, workers=4)
# labels = ['ant', 'bee', 'butterfly', 'mosquito', 'snail']
# y_pred = makeRoc(X_testzero, y_tlabhot, labels, lstm, outputSuffix='two-layer')
# y_keras = lstm.predict(X_testzero, batch_size=512)
# auc_score = roc_auc_score(y_tlabhot, y_keras)
# print(auc_score)
# model.save('./lstm/Quickdraw5Class1.h5')

In [21]:
# learningCurve(history)

## GRU Floating Point Model Training

In [22]:
# adam = Adam(lr = 0.001)
# gru.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
# history = gru.fit(array1_shuffled, array2_shuffled, batch_size = 512, epochs = 50, 
#                     validation_split = 0.2, shuffle = True, callbacks = None,
#                     use_multiprocessing=True, workers=4)
# labels = ['ant', 'bee', 'butterfly', 'mosquito', 'snail']
# y_pred = makeRoc(X_testzero, y_tlabhot, labels, gru, outputSuffix='two-layer')
# y_keras = gru.predict(X_testzero, batch_size=512)
# auc_score = roc_auc_score(y_tlabhot, y_keras)
# print(auc_score)
# model.save('./gru/Quickdraw5Class1.h5')

In [23]:
# learningCurve(history)

In [ ]:
# load models
gru = load_model('./gru/Quickdraw5Class1_20.h5')
lstm = load_model('./lstm/Quickdraw5Class1_edit_8020.h5')

## QLSTM Quantization Aware Training

In [35]:
for i in [2, 4]:
    for j in [2, 4, 6, 8, 10, 12, 14, 16, 18]:
        int_bits = i
        total_bits = i+j+1
        config = {
            "lstm_10":{
                "kernel_quantizer" : f"quantized_bits({total_bits},{int_bits},1)",
                 "bias_quantizer" : f"quantized_bits({total_bits},{int_bits},1)",
                 "recurrent_quantizer": f"quantized_bits({total_bits},{int_bits},1)",
                 "state_quantizer" : f"quantized_bits({total_bits},{int_bits},1)"
            },
            "lstm_11":{
                "kernel_quantizer" : f"quantized_bits({total_bits},{int_bits},1)",
                 "bias_quantizer" : f"quantized_bits({total_bits},{int_bits},1)",
                 "recurrent_quantizer": f"quantized_bits({total_bits},{int_bits},1)",
                 "state_quantizer" : f"quantized_bits({total_bits},{int_bits},1)"
            },
            "QDense":{
                "kernel_quantizer" : f"quantized_bits({total_bits},{int_bits},1)",
                "bias_quantizer" : f"quantized_bits({total_bits},{int_bits},1)"
            },
            "QConv1D":{
                "kernel_quantizer" : f"quantized_bits({total_bits},{int_bits},1)",
                "bias_quantizer" : f"quantized_bits({total_bits},{int_bits},1)"
            },
            "relu_0" : f"quantized_relu({total_bits},{int_bits},1)",
            "relu_1" : f"quantized_relu({total_bits},{int_bits},1)",
            "relu_2" : f"quantized_relu({total_bits},{int_bits},1)",
#             "QActivation":{
#                 f"quantized_bits({total_bits},{int_bits},1)"
#             }
        }
        qlstm = model_quantize(lstm, config, total_bits, transfer_weights=False)
        qlstm.summary()
        adam = Adam(lr = 0.001)
        qlstm.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
        history = qlstm.fit(array1_shuffled, array2_shuffled, batch_size = 512, epochs = 50, 
                            validation_split = 0.2, shuffle = True, callbacks = None,
                            use_multiprocessing=True, workers=4)

        y_keras = qlstm.predict(X_testzero, batch_size=512)
        auc_score = roc_auc_score(y_tlabhot, y_keras)
        print(auc_score)
        model.save(f'./qlstm_{i}int/qlstm_{j}frac.h5')

TypeError: Could not locate class 'QConv1D'. Make sure custom classes are decorated with `@keras.saving.register_keras_serializable()`. Full object config: {'module': 'keras.layers', 'class_name': 'QConv1D', 'config': {'name': 'conv1d', 'trainable': True, 'dtype': 'float32', 'filters': 128, 'kernel_size': [6], 'strides': [1], 'padding': 'valid', 'data_format': 'channels_last', 'dilation_rate': [1], 'groups': 1, 'activation': 'linear', 'use_bias': True, 'kernel_initializer': {'module': 'keras.initializers', 'class_name': 'GlorotUniform', 'config': {'seed': None}, 'registered_name': None}, 'bias_initializer': {'module': 'keras.initializers', 'class_name': 'Zeros', 'config': {}, 'registered_name': None}, 'kernel_regularizer': None, 'bias_regularizer': None, 'activity_regularizer': None, 'kernel_constraint': None, 'bias_constraint': None, 'kernel_quantizer': 'quantized_bits(5,2,1)', 'bias_quantizer': 'quantized_bits(5,2,1)'}, 'registered_name': None, 'build_config': {'input_shape': [None, 

In [ ]:
labels = ['ant', 'bee', 'butterfly', 'mosquito', 'snail']
y_pred = makeRoc(X_testzero, y_tlabhot, labels, qlstm, outputSuffix='two-layer')

## QGRU Quantization Aware Training

In [ ]:
for i in [2, 4]:
    for j in [2, 4, 6, 8, 10, 12, 14, 16, 18]:
        int_bits = i
        total_bits = i+j+1
        config = {
            "QGRU":{
                "kernel_quantizer" : f"quantized_bits({total_bits},{int_bits},1)",
                 "bias_quantizer" : f"quantized_bits({total_bits},{int_bits},1)",
                 "recurrent_quantizer": f"quantized_bits({total_bits},{int_bits},1)",
                 "state_quantizer" : f"quantized_bits({total_bits},{int_bits},1)"
            },
            
            "QDense":{
                "kernel_quantizer" : f"quantized_bits({total_bits},{int_bits},1)",
                "bias_quantizer" : f"quantized_bits({total_bits},{int_bits},1)"
            },
            "QConv1D":{
                "kernel_quantizer" : f"quantized_bits({total_bits},{int_bits},1)",
                "bias_quantizer" : f"quantized_bits({total_bits},{int_bits},1)"
            },
            "relu_0" : f"quantized_relu({total_bits},{int_bits},1)",
            "relu_1" : f"quantized_relu({total_bits},{int_bits},1)"
        }
        qgru = model_quantize(gru, config, total_bits, transfer_weights=False)
        qgru.summary()     
        
        adam = Adam(lr = 0.001)
        qgru.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
        history = qgru.fit(array1_shuffled, array2_shuffled, batch_size = 256, epochs = 50, 
                            validation_split = 0.2, shuffle = True, callbacks = None,
                            use_multiprocessing=True, workers=4)
# labels = ['ant', 'bee', 'butterfly', 'mosquito', 'snail']
# y_pred = makeRoc(X_testzero, y_tlabhot, labels, qgru, outputSuffix='two-layer')
        y_keras = qgru.predict(X_testzero, batch_size=512)
        auc_score = roc_auc_score(y_tlabhot, y_keras)
        print(auc_score)
        model.save(f'./qgru_{i}int/qgru_{j}frac.h5')

C:\Users\cyihu\anaconda3\envs\hls4ml-tutorial\lib\site-packages\keras\optimizers\optimizer_v2\adam.py:114: UserWarning: The `lr` argument is deprecated, use `learning_rate` instead.
  super().__init__(name, **kwargs)


Epoch 1/50
379/379 [==============================] - 22s 47ms/step - loss: 0.5675 - accuracy: 0.7958 - val_loss: 0.3053 - val_accuracy: 0.9122
Epoch 2/50
379/379 [==============================] - 16s 43ms/step - loss: 0.2700 - accuracy: 0.9202 - val_loss: 0.2436 - val_accuracy: 0.9272
Epoch 3/50
379/379 [==============================] - 16s 43ms/step - loss: 0.2265 - accuracy: 0.9316 - val_loss: 0.2121 - val_accuracy: 0.9350
Epoch 4/50
379/379 [==============================] - 16s 43ms/step - loss: 0.2036 - accuracy: 0.9373 - val_loss: 0.1929 - val_accuracy: 0.9396
Epoch 5/50
379/379 [==============================] - 16s 42ms/step - loss: 0.1819 - accuracy: 0.9432 - val_loss: 0.1711 - val_accuracy: 0.9459
Epoch 6/50
379/379 [==============================] - 16s 43ms/step - loss: 0.1680 - accuracy: 0.9471 - val_loss: 0.1658 - val_accuracy: 0.9480
Epoch 7/50
379/379 [==============================] - 16s 43ms/step - loss: 0.1552 - accuracy: 0.9506 - val_loss: 0.1552 - val_accuracy:

In [ ]:
labels = ['ant', 'bee', 'butterfly', 'mosquito', 'snail']
y_pred = makeRoc(X_testzero, y_tlabhot, labels, qgru, outputSuffix='two-layer')

## Check the train and test data

In [ ]:

ant = 0
bee = 0
butterfly = 0
mosquito = 0
snail = 0
for x in y_test:
  if x == 0:
    ant = ant + 1
  elif x == 1:
    bee = bee + 1
  elif x == 2:
    butterfly = butterfly+ 1
  elif x==3:
    mosquito = mosquito + 1
  elif x ==4:
    snail = snail + 1 

In [ ]:
print(ant)
print(bee)
print(butterfly)
print(mosquito)
print(snail)

2500
2500
2500
2500
2500


In [ ]:

ant = 0
bee = 0
butterfly = 0
mosquito = 0
snail = 0
for x in y_train:
  if x == 0:
    ant = ant + 1
  elif x == 1:
    bee = bee + 1
  elif x == 2:
    butterfly = butterfly+ 1
  elif x==3:
    mosquito = mosquito + 1
  elif x ==4:
    snail = snail + 1 

In [ ]:
print(ant)
print(bee)
print(butterfly)
print(mosquito)
print(snail)

110687
107059
104288
108399
119076
